In [ ]:
import pandas as pd
import numpy as np
pd

In [ ]:
import sys, importlib, pandas as pd

pkgs = [
    "pandas", "numpy", "geopandas", "matplotlib",
    "shapely", "contextily", "pyproj", "fiona", "rtree",
    "folium"
]

rows = [{"package": "python", "version": sys.version.split()[0]}]
for p in pkgs:
    try:
        m = importlib.import_module(p)
        v = getattr(m, "__version__", getattr(m, "version", "unknown"))
    except Exception:
        v = "not installed"
    rows.append({"package": p, "version": v})

df = pd.DataFrame(rows)
df
# df.to_csv("lib_versions.csv", index=False)  # 필요하면 저장


In [ ]:
df = pd.read_excel('./data/안전상비의약품.xlsx', engine='openpyxl')
df.columns

In [ ]:
df = df[['개방서비스명', '소재지전체주소', '도로명전체주소', '사업장명', '좌표정보X(EPSG5174)','좌표정보Y(EPSG5174)']]
df['소재지전체주소'].isnull().sum()

In [ ]:
import pandas as pd

pat = r'[\(（]([^()（）]+)[\)）]\s*$'
df["동"] = df['도로명전체주소'].astype("string").str.extract(pat)


import re

pat = re.compile(r'[\(（]([^()（）]+)[\)）]\s*$')

def extract_trailing_paren_safe(x):
    if not isinstance(x, str):
        return None
    m = pat.search(x)
    return m.group(1).strip() if m else None

df["동"] = df['도로명전체주소'].apply(extract_trailing_paren_safe)



In [ ]:
def take_before_comma(x):
    """문자열 x에서 ',' 앞부분(0번째)만 반환. 공백 제거. x가 NaN이면 그대로."""
    import pandas as pd
    if pd.isna(x):
        return x
    s = str(x)
    return s.split(',', 1)[0].strip()

df["동"] = df["동"].apply(take_before_comma)
df_dunsan = df[df["동"] == "둔산동"].copy()
df_dunsan.reset_index(drop=True, inplace=True)
df_dunsan.head(5)

In [ ]:
import geopandas as gpd
# df: pandas DataFrame, 5174 기준 x,y 열 보유
gdf = gpd.GeoDataFrame(
    df_dunsan.copy(),
    geometry=gpd.points_from_xy(df_dunsan['좌표정보X(EPSG5174)'], df_dunsan['좌표정보Y(EPSG5174)']),
    crs="EPSG:5174"               # 현재 좌표계 지정
)
gdf = gdf.to_crs(epsg=4326)       # WGS84로 변환

gdf['lat']= gdf.geometry.y
gdf['lon']= gdf.geometry.x

gdf.to_csv('./data/안전상비의약품_둔산동_WGS84.csv', index=False)